# Layer Transformation & Portfolio Aggregation Using RMS Output

This notebook answers two questions:

1. **How doe I take a ground-up ELT and produce a *layered* mean loss, σᵢ, and σc?**
2. **How does it then add that layered contract to an existing portfolio?**

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import norm

np.random.seed(42)
pd.options.display.float_format = '{:,.2f}'.format

## Step 1: Create the ELTs

An ELT is just a table. Each row is one simulated event. The columns are:

| Column | Meaning |
|--------|---------|
| `event_id` | Unique ID for the simulated event (e.g. a specific hurricane track) |
| `rate` | How often this event occurs per year (e.g. 0.01 = once every 100 years) |
| `mean_loss` | Expected loss **if** this event happens |
| `sigma_i` | Independent std dev — uncertainty unique to *this* risk (idiosyncratic) |
| `sigma_c` | Correlated std dev — uncertainty shared with *all* risks in this event |

We create **two ELTs**: one for the existing portfolio, one for a new contract we want to add.

In [ ]:
N = 10

event_ids = np.arange(1, N + 1)
rates     = np.array([0.10, 0.08, 0.06, 0.05, 0.04, 0.03, 0.02, 0.01, 0.005, 0.002])

# --- Existing portfolio ELT ---
existing_elt = pd.DataFrame({
    'event_id':  event_ids,
    'rate':      rates,
    'mean_loss': [200, 400, 700, 1_200, 2_000, 3_500, 5_000, 7_500, 10_000, 15_000],
    'sigma_i':   [60,  100, 180, 300,   500,   800,   1_100, 1_500, 2_000,  2_800],
    'sigma_c':   [80,  160, 280, 480,   800,  1_400,  2_000, 3_000, 4_000,  6_000],
})

# --- New contract ELT (same event IDs — same hazard catalogue) ---
new_elt = pd.DataFrame({
    'event_id':  event_ids,
    'rate':      rates,
    'mean_loss': [150, 300, 500, 900, 1_500, 2_500, 3_800, 5_500, 7_000, 11_000],
    'sigma_i':   [45,  80,  120, 220, 380,   600,   850,   1_100, 1_400, 2_100],
    'sigma_c':   [60,  120, 200, 360, 600,  1_000,  1_500, 2_200, 3_000, 4_500],
})

print("Existing Portfolio ELT (ground-up, £000s):")
print(existing_elt.to_string(index=False))
print("\nNew Contract ELT (ground-up, £000s):")
print(new_elt.to_string(index=False))

## Step 2: Fit a Lognormal to Each Event's Moments

**Quite honestly there are loads of distribution that can be fitted, e.g pareto etc.**

We don't have thousands of simulated loss samples per event, we only have **two numbers**: the mean and the total standard deviation.

We assume a **lognormal distribution** for each event's loss and fits it using those two numbers.

**Why lognormal?** Losses are always ≥ 0 and tend to be right-skewed (small losses are common, very large losses are rare). The lognormal captures this naturally.

### The total standard deviation

First we combine the two std dev components into one:

$$\sigma_{total} = \sqrt{\sigma_i^2 + \sigma_c^2}$$

> **Plain English:** σᵢ and σc are independent of each other (one is idiosyncratic, one is systematic), so their variances not their std devs — add together. We then take the square root to get back to a std dev.

### Converting to lognormal parameters

A lognormal distribution is defined by two parameters: **μ_ln** (the mean of the log) and **σ_ln** (the std dev of the log). We find them from the real-world mean and σ_total:

$$\sigma_{ln} = \sqrt{\ln\left(1 + \left(\frac{\sigma_{total}}{\mu}\right)^2\right)}$$

$$\mu_{ln} = \ln(\mu) - \frac{1}{2}\sigma_{ln}^2$$

> **Plain English:** The term (σ/μ)² is the **squared coefficient of variation**  it tells us how variable the loss is relative to its average. We plug this into the log formula to find the spread of the underlying normal distribution. Then μ_ln is just the log of the mean, adjusted downward because the lognormal mean is always pulled up by the heavy right tail.

In [ ]:
def fit_lognormal(elt: pd.DataFrame) -> pd.DataFrame:
    """
    Given an ELT with mean_loss, sigma_i, sigma_c,
    compute the lognormal parameters mu_ln and sigma_ln for each event.
    """
    df = elt.copy()

    # Total std dev: sigma_total = sqrt(sigma_i^2 + sigma_c^2)
    df['sigma_total'] = np.sqrt(df['sigma_i']**2 + df['sigma_c']**2)

    # Lognormal sigma: sigma_ln = sqrt( ln(1 + (sigma_total/mean)^2) )
    cv2 = (df['sigma_total'] / df['mean_loss'])**2
    df['sigma_ln'] = np.sqrt(np.log(1 + cv2))

    # Lognormal mu: mu_ln = ln(mean) - 0.5 * sigma_ln^2
    df['mu_ln'] = np.log(df['mean_loss']) - 0.5 * df['sigma_ln']**2

    return df


existing_fitted = fit_lognormal(existing_elt)

print("Existing ELT with fitted lognormal parameters:")
cols = ['event_id', 'mean_loss', 'sigma_total', 'mu_ln', 'sigma_ln']
print(existing_fitted[cols].to_string(index=False))

## Step 3: Apply the Reinsurance/Pricing Layer

A reinsurance layer is defined by:
- **Retention (R):** The layer only starts paying once losses exceed R
- **Limit (L):** The maximum the layer pays
- **Exhaustion (E = R + L):** The loss level at which the layer is fully used

So the layer pays: $\min(\max(\text{loss} - R,\ 0),\ L)$

We need to find the **expected value** and **variance** of this layered loss, given our lognormal distribution.

### The Limited Expected Value (LEV)

The key building block is the **LEV** — the expected value of losses *capped* at some threshold $d$:

$$E[\min(X,\ d)] = e^{\mu_{ln} + \frac{1}{2}\sigma_{ln}^2} \cdot \Phi\!\left(\frac{\mu_{ln} + \sigma_{ln}^2 - \ln d}{\sigma_{ln}}\right) + d \cdot \left[1 - \Phi\!\left(\frac{\ln d - \mu_{ln}}{\sigma_{ln}}\right)\right]$$

> **Plain English:** This splits the expectation into two pieces:
> - The **first term** is the contribution from losses *below* d — they enter at their full value, weighted by the lognormal CDF.
> - The **second term** is the contribution from losses *above* d — they are all capped at d, weighted by how often the loss exceeds d.
> 
> $\Phi$ is just the standard normal CDF — the S-shaped curve you get from a normal distribution.

---

### Layer mean loss

The layer pays between R and E, so the layer mean is just the difference of two LEVs:

$$\mu_{layer} = E[\min(X, E)] - E[\min(X, R)]$$

> **Plain English:** The expected loss capped at E, minus the expected loss capped at R. What's left is the expected loss *between* R and E — exactly what the layer pays.

---

### Layer variance

We need the second moment $E[\text{layer}^2]$ to find the variance. For a lognormal, the second moment of a capped variable is:

$$E[\min(X, d)^2] = e^{2\mu_{ln} + 2\sigma_{ln}^2} \cdot \Phi\!\left(\frac{\mu_{ln} + 2\sigma_{ln}^2 - \ln d}{\sigma_{ln}}\right) + d^2 \cdot \left[1 - \Phi\!\left(\frac{\ln d - \mu_{ln}}{\sigma_{ln}}\right)\right]$$

Since the layer = min(X, E) − min(X, R), the second moment of the layer is:

$$E[\text{layer}^2] = E[\min(X,E)^2] - 2R \cdot \left(E[\min(X,E)] - E[\min(X,R)]\right) - E[\min(X,R)^2]$$

Then variance:

$$\text{Var}[\text{layer}] = E[\text{layer}^2] - \mu_{layer}^2$$

> **Plain English:** Variance = average of the squared values, minus the square of the average. This is the standard identity $\text{Var}(X) = E[X^2] - E[X]^2$.

---

### Splitting layer σ back into σᵢ and σc

Once we have the total layer std dev, we split it back into independent and correlated components by preserving the **same ratio** as in the ground-up:

$$f_c = \frac{\sigma_c}{\sigma_{total}}, \qquad f_i = \frac{\sigma_i}{\sigma_{total}}$$

$$\sigma_{c,layer} = f_c \cdot \sigma_{total,layer}, \qquad \sigma_{i,layer} = f_i \cdot \sigma_{total,layer}$$

> **Plain English:** The layer transformation squeezes or stretches the distribution, but it doesn't change *what fraction of uncertainty is correlated vs independent*. So we assume the split stays the same and just scale both components proportionally.

In [ ]:
# -------------------------------------------------------
# Helper: Limited Expected Value for lognormal
# -------------------------------------------------------
def lev(d, mu_ln, sigma_ln):
    """E[min(X, d)] for a lognormal(mu_ln, sigma_ln)."""
    if d <= 0:
        return 0.0
    z1 = (mu_ln + sigma_ln**2 - np.log(d)) / sigma_ln
    z2 = (np.log(d) - mu_ln) / sigma_ln
    return np.exp(mu_ln + 0.5 * sigma_ln**2) * norm.cdf(z1) + d * (1 - norm.cdf(z2))


def lev2(d, mu_ln, sigma_ln):
    """E[min(X, d)^2] for a lognormal(mu_ln, sigma_ln)."""
    if d <= 0:
        return 0.0
    z1 = (mu_ln + 2 * sigma_ln**2 - np.log(d)) / sigma_ln
    z2 = (np.log(d) - mu_ln) / sigma_ln
    return np.exp(2 * mu_ln + 2 * sigma_ln**2) * norm.cdf(z1) + d**2 * (1 - norm.cdf(z2))


# -------------------------------------------------------
# Apply layer to a fitted ELT DataFrame
# -------------------------------------------------------
def apply_layer(fitted_elt: pd.DataFrame, retention: float, limit: float) -> pd.DataFrame:
    """
    Apply a reinsurance layer (retention R, limit L) to a fitted ELT.
    Returns a new DataFrame with layered mean_loss, sigma_i, sigma_c.
    """
    R = retention
    E = retention + limit   # exhaustion point

    rows = []
    for _, row in fitted_elt.iterrows():
        mu_ln    = row['mu_ln']
        sigma_ln = row['sigma_ln']

        # --- Layer mean: LEV(E) - LEV(R) ---
        layer_mean = lev(E, mu_ln, sigma_ln) - lev(R, mu_ln, sigma_ln)

        # --- Layer variance from second moments ---
        lev_R  = lev(R,  mu_ln, sigma_ln)
        lev_E  = lev(E,  mu_ln, sigma_ln)
        lev2_R = lev2(R, mu_ln, sigma_ln)
        lev2_E = lev2(E, mu_ln, sigma_ln)

        # E[layer^2] = E[min(X,E)^2] - 2R*(E[min(X,E)] - E[min(X,R)]) - E[min(X,R)^2]
        e_layer2   = lev2_E - 2 * R * (lev_E - lev_R) - lev2_R
        layer_var  = max(e_layer2 - layer_mean**2, 0.0)
        layer_sig  = np.sqrt(layer_var)

        # --- Preserve the sigma_i / sigma_c split ---
        sigma_total = row['sigma_total']
        f_c = row['sigma_c'] / sigma_total if sigma_total > 0 else 0.5
        f_i = row['sigma_i'] / sigma_total if sigma_total > 0 else 0.5

        rows.append({
            'event_id':  row['event_id'],
            'rate':      row['rate'],
            'mean_loss': layer_mean,
            'sigma_i':   f_i * layer_sig,
            'sigma_c':   f_c * layer_sig,
        })

    return pd.DataFrame(rows)


# -------------------------------------------------------
# Apply a £3,000 xs £1,000 layer
# -------------------------------------------------------
RETENTION = 1_000
LIMIT     = 3_000

existing_fitted = fit_lognormal(existing_elt)
new_fitted      = fit_lognormal(new_elt)

existing_layered = apply_layer(existing_fitted, RETENTION, LIMIT)
new_layered      = apply_layer(new_fitted,      RETENTION, LIMIT)

print(f"Layer: £{LIMIT:,} xs £{RETENTION:,}  (exhaustion = £{RETENTION+LIMIT:,})")
print()
print("Existing ELT — BEFORE layer:")
print(existing_elt[['event_id','rate','mean_loss','sigma_i','sigma_c']].to_string(index=False))
print()
print("Existing ELT — AFTER layer:")
print(existing_layered.round(1).to_string(index=False))

In [ ]:
# -------------------------------------------------------
# Plot: ground-up vs layered moments for the existing ELT
# -------------------------------------------------------
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
fig.suptitle('Ground-up vs Layered Moments — Existing ELT', fontsize=12, fontweight='bold')

events = existing_elt['event_id']

for ax, col, label in zip(
    axes,
    ['mean_loss', 'sigma_i', 'sigma_c'],
    ['Mean Loss', 'σᵢ (independent)', 'σc (correlated)']
):
    ax.plot(events, existing_elt[col],     'o-', label='Ground-up', color='steelblue')
    ax.plot(events, existing_layered[col], 's--', label='Layered',   color='darkorange')
    ax.axhline(RETENTION, color='grey', ls=':', lw=1, label=f'Retention={RETENTION:,}')
    ax.axhline(RETENTION + LIMIT, color='lightcoral', ls=':', lw=1, label=f'Exhaustion={RETENTION+LIMIT:,}')
    ax.set_title(label)
    ax.set_xlabel('Event ID')
    ax.set_ylabel('£000s')
    ax.legend(fontsize=7)

plt.tight_layout()
plt.show()

**What you can see in the chart:**

- For **low-severity events** (IDs 1–3), losses barely reach the retention, so the layered values are near zero.
- For **mid-severity events** (IDs 4–7), losses are inside the layer — these events drive the layered mean and std devs.
- For **extreme events** (IDs 8–10), losses blow through the exhaustion point. The layer pays its full limit, so the mean is capped but the std dev is *also* low — because the layer always pays the maximum regardless of how bad the event is.

---
## Step 4 — Add the New Contract to the Existing Portfolio

Now we have:
- `existing_layered` — the layered ELT for the existing portfolio
- `new_layered` — the layered ELT for the new contract

Both have the **same event IDs**. We combine them row-by-row using three simple rules:

### Rule 1: Mean losses add directly

$$\mu_{combined} = \mu_{existing} + \mu_{new}$$

> **Plain English:** Expected values are always additive. If you expect to lose £100 from one risk and £80 from another in the same event, you expect to lose £180 total. This holds regardless of correlation.


### Rule 2: Independent σᵢ adds in quadrature

$$\sigma_{i,combined} = \sqrt{\sigma_{i,existing}^2 + \sigma_{i,new}^2}$$

> **Plain English:** Independent uncertainties don't reinforce each other. If one building's damage uncertainty is £60 and another's is £45, the combined uncertainty is not £105 — it's $\sqrt{60^2 + 45^2} = £75$. The two uncertainties partially cancel because they're independent. This is *diversification at work*.

---

### Rule 3: Correlated σc adds linearly

$$\sigma_{c,combined} = \sigma_{c,existing} + \sigma_{c,new}$$

> **Plain English:** Correlated uncertainty *does not diversify*. If the wind speed across a region is uncertain, every risk in that region is uncertain in the *same direction*. So the correlated uncertainties just pile on top of each other — full linear addition, no offset.

In [ ]:
def add_to_portfolio(existing_layered: pd.DataFrame, new_layered: pd.DataFrame) -> pd.DataFrame:
    """
    Combine a new contract with the existing portfolio, event by event.

    Rules:
      mean_loss_combined = mean_existing + mean_new          (additive)
      sigma_i_combined   = sqrt(sigma_i_existing^2 + sigma_i_new^2)  (quadrature)
      sigma_c_combined   = sigma_c_existing + sigma_c_new             (linear)
    """
    merged = existing_layered.merge(new_layered, on=['event_id', 'rate'], suffixes=('_exist', '_new'))

    combined = pd.DataFrame()
    combined['event_id']  = merged['event_id']
    combined['rate']      = merged['rate']

    # Rule 1: means add
    combined['mean_loss'] = merged['mean_loss_exist'] + merged['mean_loss_new']

    # Rule 2: sigma_i adds in quadrature
    combined['sigma_i']   = np.sqrt(merged['sigma_i_exist']**2 + merged['sigma_i_new']**2)

    # Rule 3: sigma_c adds linearly
    combined['sigma_c']   = merged['sigma_c_exist'] + merged['sigma_c_new']

    return combined


combined_portfolio = add_to_portfolio(existing_layered, new_layered)

print("Side-by-side comparison (layered, £000s):")
print()

compare = existing_layered[['event_id','mean_loss','sigma_i','sigma_c']].copy()
compare.columns = ['event_id', 'exist_mean', 'exist_si', 'exist_sc']
compare['new_mean'] = new_layered['mean_loss'].values
compare['new_si']   = new_layered['sigma_i'].values
compare['new_sc']   = new_layered['sigma_c'].values
compare['comb_mean'] = combined_portfolio['mean_loss'].values
compare['comb_si']   = combined_portfolio['sigma_i'].values
compare['comb_sc']   = combined_portfolio['sigma_c'].values

print(compare.round(1).to_string(index=False))

In [ ]:
# -------------------------------------------------------
# Plot: visualise the aggregation rules
# -------------------------------------------------------
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Portfolio Aggregation — Existing vs New vs Combined (Layered)', fontsize=12, fontweight='bold')

events = combined_portfolio['event_id']
pairs = [
    ('mean_loss', 'Mean Loss  (additive)'),
    ('sigma_i',   'σᵢ  (quadrature — diversifies)'),
    ('sigma_c',   'σc  (linear — no diversification)'),
]

for ax, (col, title) in zip(axes, pairs):
    ax.plot(events, existing_layered[col],    'o-',  label='Existing',  color='steelblue',  lw=2)
    ax.plot(events, new_layered[col],         's-',  label='New',       color='darkorange',  lw=2)
    ax.plot(events, combined_portfolio[col],  '^--', label='Combined',  color='seagreen',    lw=2)

    # For sigma_i: also show the naive sum to illustrate diversification
    if col == 'sigma_i':
        naive_sum = existing_layered[col] + new_layered[col]
        ax.plot(events, naive_sum, 'x:', label='Naive sum (wrong)', color='firebrick', lw=1.5)

    ax.set_title(title, fontsize=10)
    ax.set_xlabel('Event ID')
    ax.set_ylabel('£000s')
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

**What the middle chart shows:**

The red dashed line is what you'd get if you wrongly added σᵢ linearly. The green combined line is *below* it — that gap is the **diversification benefit**. For independent uncertainty, adding two risks doesn't double the risk.

**What the right chart shows:**

For σc there is no gap — the combined line sits exactly on top of the naive sum. Correlated uncertainty gives you zero diversification benefit. This is the critical asymmetry in Tiger Eye's marginal pricing logic.

## Step 5: Portfolio AAL and Std Dev

With the combined portfolio ELT we can now compute two headline numbers:

### Annual Average Loss (AAL)

$$\text{AAL} = \sum_{e} \text{rate}_e \times \mu_e$$

> **Plain English:** For each event, multiply how likely it is (rate) by how much we expect to lose (mean). Then sum across all events. This is the long-run average annual loss, what you'd expect to pay out on average every year.

---

### Portfolio Standard Deviation

We use the **law of total variance** across events:

$$\text{Var}(\text{annual loss}) = \sum_{e} \text{rate}_e \cdot \left(\sigma_{total,e}^2 + \mu_e^2\right) - \text{AAL}^2$$

$$\sigma_{portfolio} = \sqrt{\text{Var}(\text{annual loss})}$$

> **Plain English:** The annual loss variance has two sources. First, *within each event* there's uncertainty around the mean (the $\sigma_{total}^2$ term). Second, *across events* the mean loss itself varies (the $\mu^2$ term, averaged across events, minus the square of the overall average). Both contribute to year-on-year volatility.

In [ ]:
def portfolio_stats(elt: pd.DataFrame, label: str) -> dict:
    """Compute AAL and portfolio std dev from a layered ELT."""
    sigma_total = np.sqrt(elt['sigma_i']**2 + elt['sigma_c']**2)

    aal = (elt['rate'] * elt['mean_loss']).sum()

    # Law of total variance
    var = (elt['rate'] * (sigma_total**2 + elt['mean_loss']**2)).sum() - aal**2
    std = np.sqrt(max(var, 0))

    print(f"{label}")
    print(f"  AAL:      £{aal:>10,.1f}")
    print(f"  Std Dev:  £{std:>10,.1f}")
    print(f"  CoV:       {std/aal:>10.3f}")
    print()
    return {'label': label, 'aal': aal, 'std': std}


stats_existing = portfolio_stats(existing_layered,    'Existing Portfolio (layered)')
stats_new      = portfolio_stats(new_layered,         'New Contract (layered)')
stats_combined = portfolio_stats(combined_portfolio,  'Combined Portfolio (layered)')

print("--- Marginal Impact of adding the New Contract ---")
print(f"  ΔAAL:     £{stats_combined['aal'] - stats_existing['aal']:>10,.1f}")
print(f"  ΔStd Dev: £{stats_combined['std'] - stats_existing['std']:>10,.1f}")

In [ ]:
# -------------------------------------------------------
# Bar chart: AAL and Std Dev before and after
# -------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
fig.suptitle('Marginal Impact of Adding New Contract', fontsize=12, fontweight='bold')

labels  = ['Existing', 'Existing +\nNew Contract']
aal_vals = [stats_existing['aal'], stats_combined['aal']]
std_vals = [stats_existing['std'], stats_combined['std']]

for ax, vals, title in zip(axes, [aal_vals, std_vals], ['AAL (£000s)', 'Portfolio Std Dev (£000s)']):
    bars = ax.bar(labels, vals, color=['steelblue', 'darkorange'], alpha=0.85, width=0.5)
    ax.bar_label(bars, fmt='£{:,.0f}', padding=3, fontsize=9)
    ax.set_title(title)
    ax.set_ylabel('£000s')

    # Arrow showing the delta
    delta = vals[1] - vals[0]
    ax.annotate('', xy=(1, vals[1]), xytext=(1, vals[0]),
                arrowprops=dict(arrowstyle='<->', color='firebrick', lw=2))
    ax.text(1.15, (vals[0] + vals[1]) / 2, f'+£{delta:,.0f}',
            color='firebrick', fontsize=9, va='center')

plt.tight_layout()
plt.show()